# NWM RouteLink file for developing topologic relationships

This notebook was adapted from original work by James Halgren (GitHub @jameshalgren).

This notebook demonstrates accessing the National Water Model (NWM) topological definition of the NWM channel routing simulation. Using these topological relationships, we can determine the distance between a reach and its nearest downstream gage.

- RouteLink_CONUS.nc sourced from https://www.nco.ncep.noaa.gov/pmb/codes/nwprod/nwm.v3.0.13/parm/domain/RouteLink_CONUS.nc. (note: the version number is not static and gets updated frequently (v3.0.xx))

Note: the source for RouteLink may change as the NWM gets updated by NOAA-OWP. Previous versions are not available for download.

Adapted by Quinn Lee (GitHub @quinnylee)

In [1]:
import xarray as xr
from collections import defaultdict
import json

In [2]:
routelink_ds = xr.open_dataset("../01_data_collection/RouteLink_CONUS.nc")

In [3]:
routelink_ds

<xarray.Dataset> Size: 269MB
Dimensions:            (feature_id: 2776734)
Coordinates:
    lon                (feature_id) float32 11MB ...
    lat                (feature_id) float32 11MB ...
Dimensions without coordinates: feature_id
Data variables: (12/21)
    link               (feature_id) int32 11MB ...
    from               (feature_id) int32 11MB ...
    to                 (feature_id) int32 11MB ...
    alt                (feature_id) float32 11MB ...
    order              (feature_id) int32 11MB ...
    Qi                 (feature_id) float32 11MB ...
    ...                 ...
    gages              (feature_id) |S15 42MB ...
    Kchan              (feature_id) int16 6MB ...
    ascendingIndex     (feature_id) int32 11MB ...
    nCC                (feature_id) float32 11MB ...
    TopWdthCC          (feature_id) float32 11MB ...
    TopWdth            (feature_id) float32 11MB ...
Attributes:
    Convention:        CF-1.6
    featureType:       timeSeries
    history:           Created Thu Sep  9 18:11:34 2021
    processing_notes:  This file was produced Thu Sep  9 16:16:38 2021 by Kev...

In [4]:
# Subset the dataset to only the columns we want

subslice = [
    "link",
    "to",
    "gages",
    "Length"
]

routelink_df = routelink_ds[subslice].to_dataframe().astype({"link": int, "to": int,})

## Create a topology
With the downloaded Route_Link, we can generate the topology of the CONUS river network

In [5]:
routelink_df = routelink_df.set_index("link")
routelink_df

,to,gages,Length,lon,lat
link,,,,,
6635572,6635570,b' ',1070.0,-96.540199,46.228783
6635590,6635600,b' ',1117.0,-96.530647,46.213486
6635598,6635636,b' ',2303.0,-96.505341,46.201508
6635622,6635620,b' ',1119.0,-96.615021,46.200523
6635626,6635624,b' ',3171.0,-96.637161,46.195522
...,...,...,...,...,...
15456832,25371895,b' ',2208.0,-74.654648,44.979626
25371895,0,b' ',1712.0,-74.648621,44.996113
15448486,0,b' ',945.0,-74.504646,44.994370


In [6]:
all_ids = routelink_df.index

In [7]:
# the weird identifier for a location without a gage
nogage = routelink_df.loc[6635572]['gages']

In [8]:
def replace_downstreams(data, downstream_col, terminal_code):
    '''If a node is above a terminal node, set the downstream id to the negative of the current node.'''
    ds0_mask = data[downstream_col] == terminal_code
    new_data = data.copy()
    new_data.loc[ds0_mask, downstream_col] = ds0_mask.index[ds0_mask]

    # Also set negative any nodes in downstream col not in data.index
    new_data.loc[~data[downstream_col].isin(data.index), downstream_col] *= -1
    return new_data

In [9]:
# Reorganize RouteLink file
routelink_df = routelink_df.sort_index()
routelink_df = replace_downstreams(routelink_df, "to", 0)

In [10]:
routelink_df

,to,gages,Length,lon,lat
link,,,,,
101,1078719,b' ',3251.000000,-94.640541,31.086876
179,181,b' ',2412.000000,-67.986412,46.022163
181,1435,b' ',442.000000,-67.998726,46.016491
183,185,b' ',112.000000,-67.998833,46.020847
185,1065,b' ',170.000000,-67.998619,46.019711
...,...,...,...,...,...
1180001800,1180001799,b' ',1905.199951,-115.953217,32.158100
1180001801,1180001800,b' ',1746.500000,-115.935883,32.162037
1180001802,1180001795,b' ',2985.800049,-115.981506,32.159569


In [11]:
distance_to_gage = {id: 0 for id in all_ids} # stores distance to nearest downstream gage in meters, NaN if no gage downstream
num_reaches_to_gage = {id: 0 for id in all_ids} # stores number of reaches to nearest downstream gage, NaN if no gage downstream

In [12]:
def distance_and_reaches_to_gage(id, distance, num_reaches):
    has_gage = routelink_df.loc[id, "gages"] != nogage
    if not has_gage:
        length = routelink_df.loc[id, "Length"]
        distance += length
        num_reaches += 1
        downstream_id = routelink_df.loc[id, "to"]
        if downstream_id < 0: # if we hit a terminal node before finding a gage
            distance = "NaN"
            num_reaches = "NaN"
        else:
            distance, num_reaches = distance_and_reaches_to_gage(downstream_id, distance, num_reaches)
    return distance, num_reaches

In [13]:

from tqdm.notebook import tqdm

# iterate with a Jupyter-friendly progress bar
for id in tqdm(all_ids, desc="Computing distance to downstream gage"):
    has_gage = routelink_df.loc[id, "gages"] != nogage
    if has_gage:
        distance_to_gage[id] = 0
        num_reaches_to_gage[id] = 0
    else:
        distance, num_reaches = distance_and_reaches_to_gage(id, 0, 0)
        distance_to_gage[id] = distance
        num_reaches_to_gage[id] = num_reaches

Computing distance to downstream gage:   0%|          | 0/2776734 [00:00<?, ?it/s]

In [14]:
with open("./distance_to_gage.json", 'w') as dist:
    json.dump(distance_to_gage, dist, indent=4)

with open("./num_reaches_to_gage.json", 'w') as nr:
    json.dump(num_reaches_to_gage, nr, indent=4)

In [25]:
numerical_distances = [value for value in distance_to_gage.values() if value != "NaN"]
numerical_num_reaches = [value for value in num_reaches_to_gage.values() if value != "NaN"]

In [27]:
print(f"Max distance (m): {max(numerical_distances)}")
print(f"Max num reaches: {max(numerical_num_reaches)}")

Max distance (m): 1397953.0
Max num reaches: 1048
